# IRFC Historical Market Data Collection

This notebook demonstrates a safe API collection workflow. Keep API credentials out of notebooks and Git history. The included CSV snapshot can be analyzed without an API key.


## 1. Install dependencies (run in a terminal if needed)

`python -m pip install pandas requests matplotlib`

In [ ]:
import os
import requests
import pandas as pd
from pathlib import Path


## 2. Configure credentials safely

Set `ALPHAVANTAGE_API_KEY` in your local environment. Do not paste a secret into this notebook or commit a `.env` file.

In [ ]:
API_KEY = os.getenv('ALPHAVANTAGE_API_KEY')
if not API_KEY:
    raise RuntimeError('Set the ALPHAVANTAGE_API_KEY environment variable before making an API request.')


## 3. Request daily time-series data

Confirm that the provider supports the requested symbol and exchange suffix for your account before relying on the response.

In [ ]:
symbol = 'IRFC.BSE'
url = 'https://www.alphavantage.co/query'
params = {
    'function': 'TIME_SERIES_DAILY',
    'symbol': symbol,
    'outputsize': 'full',
    'apikey': API_KEY,
}
response = requests.get(url, params=params, timeout=30)
response.raise_for_status()
payload = response.json()

series_key = 'Time Series (Daily)'
if series_key not in payload:
    message = payload.get('Note') or payload.get('Information') or payload.get('Error Message') or str(payload)
    raise RuntimeError(f'Provider did not return the expected time series: {message}')

market = pd.DataFrame.from_dict(payload[series_key], orient='index')
market.index = pd.to_datetime(market.index, errors='coerce')
market = market.rename(columns={
    '1. open': 'open', '2. high': 'high', '3. low': 'low',
    '4. close': 'close', '5. volume': 'volume'
})
market = market.apply(pd.to_numeric, errors='coerce')
market.index.name = 'date'
market = market.reset_index().dropna().sort_values('date')
market.head()


## 4. Validate and export


In [ ]:
required = ['date', 'open', 'high', 'low', 'close', 'volume']
market = market[required].drop_duplicates(subset='date')
market = market[(market['high'] >= market[['open','close','low']].max(axis=1)) &
                (market['low'] <= market[['open','close','high']].min(axis=1))]
output_path = Path('data') / 'IRFC_BSE_Data.csv'
output_path.parent.mkdir(parents=True, exist_ok=True)
market.to_csv(output_path, index=False, date_format='%Y-%m-%d')
print(f'Saved {len(market):,} rows to {output_path}')


## 5. Basic exploration


In [ ]:
market.info()
display(market.describe().T)
display(market.isna().sum().rename('missing_values').to_frame())
